In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
    v_measure_score)

### LOAD DATA

In [ ]:
data = load_iris()

X = data.data
y_true = data.target

### SCALE FEATURES

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

### TEST DIFFERENT NUMBERS OF CLUSTERS

In [ ]:
results = []

for k in range(2, 11):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10)

    labels = model.fit_predict(X_scaled)

### INTERNAL METRICS

In [ ]:
inertia = model.inertia_

silhouette = silhouette_score(X_scaled,labels)

davies_bouldin = (davies_bouldin_score(X_scaled,labels))

calinski_harabasz = (calinski_harabasz_score(X_scaled,labels))

### EXTERNAL METRICS

In [ ]:
ari = adjusted_rand_score(y_true,labels)

nmi = normalized_mutual_info_score(y_true,labels)

homogeneity = homogeneity_score(y_true,labels)

completeness = completeness_score(y_true,labels)

v_measure = v_measure_score(y_true,labels)

### SAVE RESULTS

In [ ]:
results.append({"K": k,
        "Inertia": inertia,
        "Silhouette": silhouette,
        "Davies_Bouldin": davies_bouldin,
        "Calinski_Harabasz":calinski_harabasz,
        "ARI": ari,
        "NMI": nmi,
        "Homogeneity":homogeneity,
        "Completeness":completeness,
        "V_measure":v_measure})

### CREATE RESULTS DATAFRAME

In [ ]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

### SELECT FINAL K

In [ ]:
final_k = 3

final_model = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10)

final_labels = (final_model.fit_predict(X_scaled))

### FINAL INTERNAL METRICS

In [ ]:
final_silhouette = silhouette_score(X_scaled,final_labels)

final_db = davies_bouldin_score(X_scaled,final_labels)

final_ch = calinski_harabasz_score(X_scaled,final_labels)

print("\nFinal Silhouette:",final_silhouette)

print("Final Davies-Bouldin:",final_db)

print("Final Calinski-Harabasz:",final_ch)

### FINAL EXTERNAL METRICS

In [ ]:
print("\nFinal ARI:",adjusted_rand_score(y_true,final_labels))

print("Final NMI:",normalized_mutual_info_score(y_true,final_labels))

print("Final Homogeneity:",homogeneity_score(y_true,final_labels))

print("Final Completeness:",completeness_score(y_true,final_labels))

print("Final V-measure:",v_measure_score(y_true,final_labels))

### CLUSTER SIZES

In [ ]:
cluster_sizes = (
    pd.Series(final_labels)
    .value_counts()
    .sort_index())

print("\nCluster Sizes:")

print(cluster_sizes)

### Cluster Profiling

In [ ]:
df["cluster"] = final_labels

profile = (df.groupby("cluster").mean(numeric_only=True))

print(profile)

### PCA VISUALIZATION

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 7))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=final_labels,
    s=50)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Final Clustering")

plt.show()

### EVALUATION PLOTS

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(12, 9))

# WCSS
axes[0, 0].plot(
    results_df["K"],
    results_df["Inertia"],
    marker="o")
axes[0, 0].set_title("Elbow / WCSS")

# Silhouette
axes[0, 1].plot(
    results_df["K"],
    results_df["Silhouette"],
    marker="o")
axes[0, 1].set_title("Silhouette Score")

# Davies-Bouldin
axes[1, 0].plot(
    results_df["K"],
    results_df["Davies_Bouldin"],
    marker="o")
axes[1, 0].set_title("Davies-Bouldin Index")

# ARI
axes[1, 1].plot(
    results_df["K"],
    results_df["ARI"],
    marker="o")
axes[1, 1].set_title("Adjusted Rand Index")

plt.tight_layout()
plt.show()